In [2]:
import pandas as pd
import numpy as np

# -----------------------------
# Load CSV
# -----------------------------
csv_path = "Model_D_val_rollout_timing_stats.csv"
df = pd.read_csv(csv_path)

print("Loaded shape:", df.shape)
display(df.head())

# -----------------------------
# Basic counts
# -----------------------------
n_total = len(df)
n_full = int(df["is_full_rollout"].sum())
n_nonfull = n_total - n_full

print("\n==============================")
print("DATASET COUNTS")
print("==============================")
print(f"Total validation sims:   {n_total}")
print(f"Full rollouts:          {n_full}")
print(f"Non-full rollouts:      {n_nonfull}")

# -----------------------------
# Accuracy metrics
# -----------------------------
abs_err = np.abs(df["pred_area_gain_cm2"] - df["true_area_gain_cm2"])
sq_err = (df["pred_area_gain_cm2"] - df["true_area_gain_cm2"])**2

# relative error only where true > 0
mask_pos_true = df["true_area_gain_cm2"] > 0
rel_err = np.full(len(df), np.nan)
rel_err[mask_pos_true] = abs_err[mask_pos_true] / df.loc[mask_pos_true, "true_area_gain_cm2"]

df["abs_err_cm2"] = abs_err
df["sq_err_cm2"] = sq_err
df["rel_err"] = rel_err

mae = abs_err.mean()
rmse = np.sqrt(sq_err.mean())
bias = (df["pred_area_gain_cm2"] - df["true_area_gain_cm2"]).mean()
med_abs_err = np.median(abs_err)

print("\n==============================")
print("AREA GAIN PREDICTION SUMMARY")
print("==============================")
print(f"MAE [cm^2]:                  {mae:.6f}")
print(f"RMSE [cm^2]:                 {rmse:.6f}")
print(f"Median abs err [cm^2]:       {med_abs_err:.6f}")
print(f"Bias (pred - true) [cm^2]:   {bias:.6f}")

if mask_pos_true.any():
    print(f"Mean relative error:         {np.nanmean(rel_err):.6f}")
    print(f"Median relative error:       {np.nanmedian(rel_err):.6f}")

# -----------------------------
# Timing summary: all sims
# -----------------------------
print("\n==============================")
print("TIMING SUMMARY (ALL SIMS)")
print("==============================")
print(f"Mean total time [s]:         {df['total_time_sec'].mean():.6f}")
print(f"Std total time [s]:          {df['total_time_sec'].std():.6f}")
print(f"Min total time [s]:          {df['total_time_sec'].min():.6f}")
print(f"Max total time [s]:          {df['total_time_sec'].max():.6f}")

print(f"Mean setup time [s]:         {df['setup_time_sec'].mean():.6f}")
print(f"Mean loop time [s]:          {df['loop_time_sec'].mean():.6f}")
print(f"Mean plot time [s]:          {df['plot_time_sec'].mean():.6f}")
print(f"Mean time/step [s]:          {df['time_per_step_sec'].mean():.6f}")
print(f"Median time/step [s]:        {df['time_per_step_sec'].median():.6f}")

# -----------------------------
# Timing summary: full rollouts only
# -----------------------------
df_full = df[df["is_full_rollout"] == True].copy()

print("\n==============================")
print("TIMING SUMMARY (FULL ROLLOUTS ONLY)")
print("==============================")
if len(df_full) > 0:
    print(f"Number of full rollouts:     {len(df_full)}")
    print(f"Mean total time [s]:         {df_full['total_time_sec'].mean():.6f}")
    print(f"Std total time [s]:          {df_full['total_time_sec'].std():.6f}")
    print(f"Mean setup time [s]:         {df_full['setup_time_sec'].mean():.6f}")
    print(f"Mean loop time [s]:          {df_full['loop_time_sec'].mean():.6f}")
    print(f"Mean time/step [s]:          {df_full['time_per_step_sec'].mean():.6f}")
    print(f"Median time/step [s]:        {df_full['time_per_step_sec'].median():.6f}")
else:
    print("No full rollouts found.")

# -----------------------------
# Rollout-length grouped summary
# -----------------------------
group_cols = [
    "true_area_gain_cm2",
    "pred_area_gain_cm2",
    "abs_err_cm2",
    "total_time_sec",
    "setup_time_sec",
    "loop_time_sec",
    "time_per_step_sec",
]

summary_by_steps = (
    df.groupby("rollout_n_steps")[group_cols]
      .agg(["count", "mean", "std", "median"])
)

print("\n==============================")
print("SUMMARY BY ROLLOUT LENGTH")
print("==============================")
display(summary_by_steps)

# -----------------------------
# Correlation checks
# -----------------------------
print("\n==============================")
print("CORRELATIONS")
print("==============================")
corr_total_steps = df["total_time_sec"].corr(df["rollout_n_steps"])
corr_loop_steps = df["loop_time_sec"].corr(df["rollout_n_steps"])
corr_err_true = df["abs_err_cm2"].corr(df["true_area_gain_cm2"])

print(f"Corr(total_time, n_steps):   {corr_total_steps:.6f}")
print(f"Corr(loop_time, n_steps):    {corr_loop_steps:.6f}")
print(f"Corr(abs_err, true_gain):    {corr_err_true:.6f}")

# -----------------------------
# Worst-case sims
# -----------------------------
worst_abs = df.sort_values("abs_err_cm2", ascending=False).head(10)
slowest = df.sort_values("total_time_sec", ascending=False).head(10)

print("\n==============================")
print("WORST 10 SIMS BY ABSOLUTE ERROR")
print("==============================")
display(worst_abs[[
    "sim_id", "true_area_gain_cm2", "pred_area_gain_cm2",
    "abs_err_cm2", "rel_err", "rollout_n_steps"
]])

print("\n==============================")
print("SLOWEST 10 SIMS")
print("==============================")
display(slowest[[
    "sim_id", "total_time_sec", "setup_time_sec",
    "loop_time_sec", "time_per_step_sec", "rollout_n_steps"
]])

Loaded shape: (185, 10)


,sim_id,true_area_gain_cm2,pred_area_gain_cm2,total_time_sec,setup_time_sec,loop_time_sec,plot_time_sec,rollout_n_steps,time_per_step_sec,is_full_rollout
0,210,0.000000,0.000000,0.135875,0.057123,0.078641,0.000110,30,0.002621,False
1,188,26.026976,34.861504,0.235419,0.056753,0.178512,0.000153,68,0.002625,True
2,249,3.404182,7.515919,0.190055,0.066058,0.123848,0.000148,60,0.002064,False
3,7,29.051963,45.098021,0.200115,0.062370,0.137585,0.000160,68,0.002023,True
4,584,15.735928,18.120619,0.119237,0.058666,0.060466,0.000106,30,0.002016,False



DATASET COUNTS
Total validation sims:   185
Full rollouts:          30
Non-full rollouts:      155

AREA GAIN PREDICTION SUMMARY
MAE [cm^2]:                  4.215975
RMSE [cm^2]:                 6.153358
Median abs err [cm^2]:       2.909424
Bias (pred - true) [cm^2]:   3.818587
Mean relative error:         3.866227
Median relative error:       0.318289

TIMING SUMMARY (ALL SIMS)
Mean total time [s]:         0.150746
Std total time [s]:          0.034123
Min total time [s]:          0.096746
Max total time [s]:          0.235419
Mean setup time [s]:         0.061020
Mean loop time [s]:          0.089601
Mean plot time [s]:          0.000124
Mean time/step [s]:          0.002064
Median time/step [s]:        0.002030

TIMING SUMMARY (FULL ROLLOUTS ONLY)
Number of full rollouts:     30
Mean total time [s]:         0.200777
Std total time [s]:          0.007600
Mean setup time [s]:         0.060313
Mean loop time [s]:          0.140305
Mean time/step [s]:          0.002063
Median time/st

true_area_gain_cm2                                   \
                             count       mean        std     median   
rollout_n_steps                                                       
20                              31   0.579173   1.782951   0.000000   
30                              37   3.748035   6.835248   0.310188   
40                              34  10.022981  10.774864   5.650608   
50                              25  26.215533  16.493421  24.051947   
60                              28  35.287938  18.299338  35.843745   
68                              30  62.156682  30.935981  64.893989   

                pred_area_gain_cm2                                   \
                             count       mean        std     median   
rollout_n_steps                                                       
20                              31   1.069703   2.373589   0.002856   
30                              37   5.998052   8.886440   2.375242   
40                              34  14.301949  12.316106   9.783937   
50                              25  30.696325  17.612848  27.851101   
60                              28  36.825019  17.525079  36.325876   
68                              30  72.404638  32.243090  68.465008   

                abs_err_cm2             ... setup_time_sec            \
                      count       mean  ...            std    median   
rollout_n_steps                         ...                            
20                       31   0.490529  ...       0.003192  0.060933   
30                       37   2.250016  ...       0.003200  0.059720   
40                       34   4.278967  ...       0.003247  0.060268   
50                       25   4.480793  ...       0.002459  0.059547   
60                       28   3.040523  ...       0.004063  0.061333   
68                       30  11.295302  ...       0.002535  0.059806   

                loop_time_sec                               time_per_step_sec  \
                        count      mean       std    median             count   
rollout_n_steps                                                                 
20                         31  0.042060  0.003313  0.040714                31   
30                         37  0.062672  0.004314  0.061032                37   
40                         34  0.082001  0.001831  0.081212                34   
50                         25  0.101800  0.001886  0.101090                25   
60                         28  0.121833  0.001689  0.121259                28   
68                         30  0.140305  0.007547  0.139383                30   

                                               
                     mean       std    median  
rollout_n_steps                                
20               0.002103  0.000166  0.002036  
30               0.002089  0.000144  0.002034  
40               0.002050  0.000046  0.002030  
50               0.002036  0.000038  0.002022  
60               0.002031  0.000028  0.002021  
68               0.002063  0.000111  0.002050  

[6 rows x 28 columns]


CORRELATIONS
Corr(total_time, n_steps):   0.986288
Corr(loop_time, n_steps):    0.992810
Corr(abs_err, true_gain):    0.647665

WORST 10 SIMS BY ABSOLUTE ERROR


,sim_id,true_area_gain_cm2,pred_area_gain_cm2,abs_err_cm2,rel_err,rollout_n_steps
114,847,112.163036,136.147019,23.983983,0.213831,68
134,36,100.480715,123.892362,23.411646,0.232996,68
43,749,43.808899,64.501548,20.692649,0.472339,68
80,785,79.161330,96.572521,17.411191,0.219946,68
88,699,31.324138,47.591150,16.267011,0.519312,68
3,7,29.051963,45.098021,16.046057,0.552323,68
180,13,71.919488,87.715913,15.796425,0.219640,68
38,352,43.612082,59.253333,15.641251,0.358645,68
94,663,103.982249,119.519483,15.537234,0.149422,68
48,880,24.406694,38.237327,13.830632,0.566674,68



SLOWEST 10 SIMS


,sim_id,total_time_sec,setup_time_sec,loop_time_sec,time_per_step_sec,rollout_n_steps
1,188,0.235419,0.056753,0.178512,0.002625,68
17,200,0.206348,0.064386,0.141801,0.002085,68
16,580,0.205806,0.065899,0.139755,0.002055,68
48,880,0.204924,0.064292,0.140476,0.002066,68
56,657,0.204171,0.063926,0.140089,0.002060,68
37,145,0.204076,0.063707,0.140210,0.002062,68
28,172,0.203633,0.063419,0.140054,0.002060,68
40,458,0.202591,0.062115,0.140324,0.002064,68
180,13,0.202561,0.058596,0.143812,0.002115,68
38,352,0.202326,0.062804,0.139365,0.002049,68


In [4]:
import pandas as pd

df = pd.read_csv("Model_D_val_rollout_timing_stats.csv")

mean_time = df["total_time_sec"].mean()
std_time = df["total_time_sec"].std()

print(f"Mean total rollout time [s]: {mean_time:.6f}")
print(f"Std dev [s]:                {std_time:.6f}")

Mean total rollout time [s]: 0.150746
Std dev [s]:                0.034123
